# 05 · MapReduce: costo diario por organización y servicio
Implementación manual de split → map → combiner → shuffle → reduce para construir el mart `org_daily_usage_by_service`.

In [1]:
#split

from pathlib import Path
from collections import defaultdict
import json, glob, zlib
import pandas as pd

LANDING = Path('../data/datalake/landing')
archivos = sorted(glob.glob(str(LANDING / 'usage_events_stream' / '*.jsonl')))

bloques = []
for a in archivos:
    with open(a) as f:
        bloques.append([json.loads(linea) for linea in f])

print(f'{len(bloques)} bloques · {sum(len(b) for b in bloques):,} eventos')

120 bloques · 43,200 eventos


In [2]:
#map

def a_numero(x):
    """value puede llegar como número, como texto o nulo."""
    try:
        return float(x)
    except (TypeError, ValueError):
        return 0.0

def mapper(evento):
    clave = (evento['org_id'], evento['timestamp'][:10], evento['service'])   # (org, día, servicio)
    valor = {
        'costo_usd':        evento['cost_usd_increment'],
        'requests':         a_numero(evento['value']) if evento['metric'] == 'requests' else 0.0,
        'cpu_hours':        a_numero(evento['value']) if evento['metric'] == 'cpu_hours' else 0.0,
        'storage_gb_hours': a_numero(evento['value']) if evento['metric'] == 'storage_gb_hours' else 0.0,
        'eventos':          1,
        'negativos':        1 if evento['cost_usd_increment'] < 0 else 0,
    }
    return clave, valor

mapper(bloques[0][0])

(('org_cvs4f8cg', '2025-08-17', 'networking'),
 {'costo_usd': 0.9561,
  'requests': 0.0,
  'cpu_hours': 0.0,
  'storage_gb_hours': 0.0,
  'eventos': 1,
  'negativos': 0})

In [4]:
#Shuffle (cada clave va a un reducer)

REDUCERS = 4

def particionar(clave):
    return zlib.crc32(str(clave).encode()) % REDUCERS

buzones = defaultdict(list)          # reducer → lista de (clave, valor)
for parcial in parciales:
    for clave, valor in parcial.items():
        buzones[particionar(clave)].append((clave, valor))

for r in range(REDUCERS):
    print(f'reducer {r}: {len(buzones[r]):,} pares')

reducer 0: 10,857 pares
reducer 1: 10,567 pares
reducer 2: 10,535 pares
reducer 3: 10,613 pares


In [7]:
#Combiner (cada bloque suma lo suyo)

def sumar(a, b):
    return {k: a[k] + b[k] for k in a}

def combiner(bloque):
    parcial = {}
    for evento in bloque:
        clave, valor = mapper(evento)
        parcial[clave] = sumar(parcial[clave], valor) if clave in parcial else valor
    return parcial

parciales = [combiner(b) for b in bloques]
pares_sin = sum(len(b) for b in bloques)
pares_con = sum(len(p) for p in parciales)
print(f'Pares que viajarían SIN combiner: {pares_sin:,}')
print(f'Pares que viajan CON combiner:    {pares_con:,}  ({1 - pares_con / pares_sin:.1%} menos)')

Pares que viajarían SIN combiner: 43,200
Pares que viajan CON combiner:    42,572  (1.5% menos)


El combiner casi no reduce en la muestra porque los micro-lotes están desordenados y son chicos. A escala real, o si los datos llegan ordenados por fecha, la reducción es grande. Por eso lo incluimos en el diseño igual.

In [5]:
# Reduce

def reducer(pares):
    total = {}
    for clave, valor in pares:
        total[clave] = sumar(total[clave], valor) if clave in total else valor
    return total

resultado = {}
for r in range(REDUCERS):
    resultado.update(reducer(buzones[r]))

mart = pd.DataFrame([{'org_id': k[0], 'usage_date': k[1], 'service': k[2], **v}
                     for k, v in resultado.items()]).sort_values(['usage_date', 'org_id', 'service'])
print(f'{len(mart):,} filas en org_daily_usage_by_service')
mart.head(10)

11,050 filas en org_daily_usage_by_service


,org_id,usage_date,service,costo_usd,requests,cpu_hours,storage_gb_hours,eventos,negativos
9362,org_0lvsnujz,2025-07-03,analytics,8.8739,116.0,4.6168,12.7808,3,0
4363,org_0lvsnujz,2025-07-03,database,0.5723,0.0,1.7515,14.7891,3,0
1204,org_0lvsnujz,2025-07-03,storage,5.5542,255.0,0.0000,4.6668,3,0
4499,org_0lzjjege,2025-07-03,compute,12.6064,120.0,0.0000,25.5097,3,0
7197,org_0lzjjege,2025-07-03,networking,3.5227,327.0,0.0000,0.0000,3,0
1812,org_0lzjjege,2025-07-03,storage,0.4907,0.0,0.0000,21.1437,3,0
1722,org_1n13jcat,2025-07-03,compute,12.6074,116.0,0.0000,19.1756,3,0
4539,org_1n13jcat,2025-07-03,storage,7.4073,372.0,0.0000,0.0000,3,0
3302,org_1swjckjl,2025-07-03,compute,28.0672,332.0,0.0000,0.0000,3,0
3296,org_1swjckjl,2025-07-03,database,27.0970,467.0,7.1702,35.5894,12,0


In [6]:
# verificacion vs pandas

ev = pd.concat([pd.DataFrame(b) for b in bloques], ignore_index=True)
ev['value_num'] = pd.to_numeric(ev['value'], errors='coerce').fillna(0)
ev['usage_date'] = ev['timestamp'].str[:10]

control = ev.groupby(['org_id', 'usage_date', 'service']).agg(
    costo_usd=('cost_usd_increment', 'sum'), eventos=('event_id', 'size')).reset_index()

print('Filas   MapReduce:', len(mart), '| pandas:', len(control))
print('Costo   MapReduce:', round(mart.costo_usd.sum(), 2), '| pandas:', round(control.costo_usd.sum(), 2))
print('Eventos MapReduce:', mart.eventos.sum(), '| pandas:', control.eventos.sum())

EVIDENCIA = Path('../evidence'); EVIDENCIA.mkdir(exist_ok=True)
mart.to_csv(EVIDENCIA / 'org_daily_usage_by_service.csv', index=False)

Filas   MapReduce: 11050 | pandas: 11050
Costo   MapReduce: 147433.98 | pandas: 147433.98
Eventos MapReduce: 43200 | pandas: 43200



## Conclusiones
- **Resultado:** 11.050 filas (org × día × servicio), 147.433,98 USD y 43.200 eventos: coincide exactamente con `groupby` de pandas.
- **Combiner:** reduce solo un 1,5 % (43.200 → 42.572 pares) porque cada micro-lote mezcla los 60 días y casi no repite claves. A escala real (bloques de 128 MB, cientos de miles de eventos) la reducción sería de órdenes de magnitud.
- **Shuffle:** los 4 reducers reciben entre 10.535 y 10.857 pares (< 3 % de diferencia): no hay data skew con la clave (org, día, servicio).